In [2]:
import pandas as pd
import json
import unidecode
import re
import os
from pandas.io.parquet import to_parquet

In [3]:
# ================================================================
# Função para gerar a chave [titulo do filme + ano de lançamento]
# ================================================================
def generate_id_movie(title, release_year):
    title = str(title) if pd.notnull(title) else ""
    try:
        release_year = str(int(release_year))
    except:
        release_year = "0000"
    
    clear_title = title.lower().strip()
    clear_title = unidecode.unidecode(clear_title)
    clear_title = re.sub(r'[^a-z0-9]', '', clear_title)
    return f"{clear_title}_{release_year}"



# ===========================================
# Abrindo os .json em DataFrame para alterar
# ===========================================
base_path = '../data' 

with open(f'{base_path}/movies_master_completo.json', 'r', encoding='utf-8') as f:
    data_master = pd.DataFrame(json.load(f))

with open(f'{base_path}/synopsis.json', 'r', encoding='utf-8') as f:
    data_synopsis = pd.DataFrame(json.load(f))

with open(f'{base_path}/critics.json', 'r', encoding='utf-8') as f:
    data_critics = pd.DataFrame(json.load(f))



# ====================================================================================================
# Para evitar o viés de filmes com nota alta mas poucos votos, será usada a MÉDIA BAYESIANA PONDERADA
# ====================================================================================================

# O dataset será dividido entre longas e curtas já que a diferença de votos entre os grupos é gritante

# W = (v/(v+m) * R) + (m/(v+m) * C)

def weighted_rating(df_input):
    groups = df_input.groupby('type')
    result = []

    for g_name, df_group in groups:
        # C = Média Global
        C = df_group['rating'].mean()
        # m = Mínimo de votos para ser considerado 25% mais votados
        m = df_group['vote_count'].quantile(0.25)

        print(f"Grupo {g_name}: Média Global (C)={C:.2f}, Corte de Votos (m)={m:.0f}")

        def weighted_formula(row):
            v = row['vote_count']
            R = row['rating']
            return (v/(v+m) * R) + (m/(v+m) * C)
        
        df_group = df_group.copy()
        df_group['balanced_rating'] = df_group.apply(weighted_formula, axis=1)
        result.append(df_group)

    return pd.concat(result)


data_master = weighted_rating(data_master)

data_master = data_master.drop('rating', axis=1)


# Gerar IDs para todos os dataframes é crucial para garantir que o merge funcione
data_master['id_movie'] = data_master.apply(lambda x: generate_id_movie(x['title'], x['release_year']), axis=1)
data_synopsis['id_movie'] = data_synopsis.apply(lambda x: generate_id_movie(x['title'], x['release_year']), axis=1)
data_critics['id_movie'] = data_critics.apply(lambda x: generate_id_movie(x['title'], x['release_year']), axis=1)



# =========================================================================================================================================================
# Função para normalizar as colunas que possam ser string ou lista em apenas lista (o .parquet retorna erro se tiver mais de uma tipo para a mesma coluna)
# =========================================================================================================================================================

def normalize_to_list(val):
    if val is None:
        return []
    if isinstance(val, list):
        return val
    return [val]

# Colunas que devem ser listas no Master
cols_master_list = ['directed by', 'genre', 'award_category'] 

for col in cols_master_list:
    if col in data_master.columns:
        data_master[col] = data_master[col].apply(normalize_to_list)


if 'critics' in data_critics.columns:
    data_critics['critics'] = data_critics['critics'].apply(normalize_to_list)


# ============================================
# Salvar aquivos normalizados individualmente
# ============================================

file_master = f'{base_path}/dataset_master_treated.parquet'
data_master.to_parquet(file_master, index=False)

file_critics = f'{base_path}/dataset_critics_treated.parquet'
data_critics.to_parquet(file_critics, index=False)

file_synopsis = f'{base_path}/dataset_synopsis_treated.parquet'
data_synopsis.to_parquet(file_synopsis, index=False)


# ==========================================================
# Merge de todos os arquivos (master + sinopses + críticas)
# ==========================================================

df_final = data_master.copy()

if 'synopsis' in data_synopsis.columns:
    df_final = df_final.merge(
        data_synopsis[['id_movie', 'synopsis']], 
        on='id_movie', 
        how='left'
    )

if 'critics' in data_critics.columns:
    df_final = df_final.merge(
        data_critics[['id_movie', 'critics']], 
        on='id_movie', 
        how='left'
    )

# =======================
# Salvar o dataset final
# =======================

output_file = f'{base_path}/dataset_gramado_completo.parquet'
df_final.to_parquet(output_file, index=False)


Grupo cm: Média Global (C)=3.56, Corte de Votos (m)=41
Grupo lm: Média Global (C)=3.28, Corte de Votos (m)=184
